<a href="https://colab.research.google.com/github/AsmaAssa2471/my-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsmaAssa2471/my-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis + Time Window
* **Grain (One Row):** One row represents the daily performance metrics (`gsc_impressions`, `gsc_clicks`, `ga4_sessions`, etc.) for a single unique content item (`content_hash_id`) belonging to a pseudonymized client (`client_hash_id`) on a specific date (`report_date`).
* **Time Window:** Mid-panel training window of **March 2026** (`month = '2026-03'`). *Note: The final month (`2026-06`) is strictly reserved as a sealed test window.*

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
# Section 2: Verification Queries (Grain, Span, and Availability)
import duckdb
from google.colab import userdata

# 1. Retrieve Token and Initialize DuckDB
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()

# 2. Load httpfs extension and set HF Authentication via CREATE SECRET
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_auth (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("=" * 60)
print("1. GRAIN CHECK (Total Rows vs Unique Composite Key Count)")
print("=" * 60)
grain_query = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT (client_hash_id || '_' || content_hash_id || '_' || CAST(report_date AS VARCHAR))) as unique_composite_keys
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print(con.sql(grain_query).df())

print("\n" + "=" * 60)
print("2. ROW COUNT & DATE SPAN CHECK (March 2026 Mid-Panel Month)")
print("=" * 60)
span_query = """
SELECT
    COUNT(*) as total_march_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print(con.sql(span_query).df())

print("\n" + "=" * 60)
print("3. AVAILABILITY CHECK (Filtered with `is_active IS TRUE`)")
print("=" * 60)

# Query attempting both single parquet and wildcard patterns for dim_clients
avail_query = """
SELECT
    c.is_active,
    COUNT(f.content_hash_id) as surviving_row_count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients/*.parquet') c
  ON f.client_hash_id = c.client_hash_id
WHERE c.is_active IS TRUE
GROUP BY c.is_active
"""

try:
    print(con.sql(avail_query).df())
except Exception:
    # Fallback in case dim_clients is stored as a direct single parquet file
    avail_query_fallback = """
    SELECT
        c.is_active,
        COUNT(f.content_hash_id) as surviving_row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet') c
      ON f.client_hash_id = c.client_hash_id
    WHERE c.is_active IS TRUE
    GROUP BY c.is_active
    """
    print(con.sql(avail_query_fallback).df())

1. GRAIN CHECK (Total Rows vs Unique Composite Key Count)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_composite_keys
0     9841378                9841378

2. ROW COUNT & DATE SPAN CHECK (March 2026 Mid-Panel Month)
   total_march_rows start_date   end_date
0           9841378 2026-03-01 2026-03-31

3. AVAILABILITY CHECK (Filtered with `is_active IS TRUE`)
   is_active  surviving_row_count
0       True              7864344


| Bucket | Fields Included | Justification / Definition |
| :--- | :--- | :--- |
| **Features** | `gsc_impressions`, `gsc_clicks`, `ga4_sessions`, `ga4_engagement_time`, `avg_position` | Historical performance signals available at the decision moment. |
| **Label / Target** | `ctr_deficit_flag` or `traffic_decay_pct` | The target value to predict/flag for automated content refreshes. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Entity IDs and timestamps required for joining and grouping. |
| **Excluded** | `rare_tail_queries`, inactive accounts (`is_active = FALSE`), future month records (`2026-06`) | **Why:** Queries <10 impressions add noise; inactive clients distort baseline distributions; future dates cause severe data leakage. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Question 3: Verify with queries (Grain, Counts, Windows, Missing Values)
import duckdb
from google.colab import userdata

# Setup connection and secret
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_auth (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 1. GRAIN CHECK
print("=" * 60)
print("1. GRAIN VERIFICATION (One Row = Unique Composite Key)")
print("=" * 60)
print(con.sql("""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT (client_hash_id || '_' || content_hash_id || '_' || CAST(report_date AS VARCHAR))) as unique_grain_keys
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df())

# 2. ROW COUNT & TIME WINDOW CHECK
print("\n" + "=" * 60)
print("2. TIME WINDOW & ROW COUNTS VERIFICATION")
print("=" * 60)
print(con.sql("""
SELECT
    COUNT(*) as total_march_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df())

# 3. AVAILABILITY CHECK
print("\n" + "=" * 60)
print("3. AVAILABILITY VERIFICATION (Active Clients Only)")
print("=" * 60)
print(con.sql("""
SELECT
    c.is_active,
    COUNT(f.content_hash_id) as surviving_row_count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet') c
  ON f.client_hash_id = c.client_hash_id
WHERE c.is_active IS TRUE
GROUP BY c.is_active
""").df())

# 4. MISSING VALUES CHECK
print("\n" + "=" * 60)
print("4. MISSING VALUES VERIFICATION (Null Counts)")
print("=" * 60)
print(con.sql("""
SELECT
    COUNT(*) - COUNT(client_hash_id) as null_clients,
    COUNT(*) - COUNT(content_hash_id) as null_contents,
    COUNT(*) - COUNT(report_date) as null_dates,
    COUNT(*) - COUNT(gsc_impressions) as null_impressions,
    COUNT(*) - COUNT(gsc_clicks) as null_clicks,
    COUNT(*) - COUNT(ga4_sessions) as null_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df())

1. GRAIN VERIFICATION (One Row = Unique Composite Key)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_grain_keys
0     9841378            9841378

2. TIME WINDOW & ROW COUNTS VERIFICATION
   total_march_rows start_date   end_date
0           9841378 2026-03-01 2026-03-31

3. AVAILABILITY VERIFICATION (Active Clients Only)
   is_active  surviving_row_count
0       True              7864344

4. MISSING VALUES VERIFICATION (Null Counts)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   null_clients  null_contents  null_dates  null_impressions  null_clicks  \
0             0              0           0                 0            0   

   null_sessions  
0        3018741  


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits
* **What this data can NEVER tell you:**
  1. **Unbalanced History:** Clients onboarded at different times (`gsc_data_start` vs. `ga4_data_start`) mean early periods lack complete cross-channel parity.
  2. **GSC-Only Early Rows:** Early historical rows contain GSC impressions/clicks but lack GA4 engagement metrics prior to GA4 integration.
  3. **Window Overlaps & Untracked Offline Conversions:** The warehouse measures digital SERP visibility and on-page sessions, but cannot capture off-site conversions, brand sentiment, or untracked user channels.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.